In [ ]:
# Install/load
install.packages(c("MatchIt", "cobalt", "dplyr", "readr")) # run once
library(MatchIt)
library(cobalt)
library(dplyr)
library(MatchIt)
library(readr)

In [ ]:
#### read in case and control files

In [ ]:

#Pilot cases
covid <- read_csv("ns_vax_cases/covid_case_cohort.csv") 
men <- read_csv("ns_cases/viral_meningitis_case_cohort.csv")
hpv <- read_csv("ns_vax_cases/verruca_vulgaris_case_cohort.csv")
flu <- read_csv("ns_vax_cases/influenza_case_cohort.csv")
polio <- read_csv("ns_vax_cases/acute_poliomyelitis_case_cohort.csv")

hiv <- read_csv("ns_cases/asymptomatic_human_immunodeficiency_virus_infection_case_cohort.csv")

#pilot controls 
covid_ctrl <- read_csv("ns_vax_controls/covid_controls.csv")
men_ctrl <- read_csv("ns_controls/viral_meningitis_controls.csv")
hpv_ctrl <- read_csv("ns_vax_controls/verruca_vulgaris_controls.csv")
flu_ctrl <- read_csv("ns_vax_controls/influenza_controls.csv")
polio_ctrl <-  read_csv("ns_vax_controls/acute_poliomyelitis_controls.csv")

hiv_ctrl <- read_csv("ns_controls/asymptomatic_human_immunodeficiency_virus_infection_controls.csv")

In [ ]:
hiv_ctrl

In [ ]:
#### Merge case and controls in subset dataframes for each cohort

In [ ]:
# Make sure you have dplyr installed and loaded
# install.packages("dplyr")
# library(dplyr)

prepare_case_control_data <- function(cases_df, 
                                      controls_df, 
                                      col_sex = "sex_at_birth",
                                      col_ancestry = "updated_race",
                                      col_age = "age") {
  
  # 1) Harmonize column names and add case/control indicator
  # Using dplyr::mutate to be explicit
  cases <- dplyr::mutate(cases_df, case = 1L)
  controls <- dplyr::mutate(controls_df, case = 0L)

  # 2) Bind, clean types, drop missing
  df <- dplyr::bind_rows(cases, controls)

  # 3) Mutate new columns using the .data pronoun
  # This lets us use the string variables (col_sex, etc.)
  df <- df |>
    dplyr::mutate(
      sex = factor(.data[[col_sex]]),
      ancestry = factor(.data[[col_ancestry]]),
      age = as.numeric(.data[[col_age]])
    )

  # 4) Quick sanity checks (prints to the console)
  message("--- Sanity Checks ---")
  message("Cases vs. Controls:")
  print(table(df$case))
  
  message("Sex vs. Case:")
  print(table(df$sex, df$case))
  
  message("Ancestry vs. Case:")
  print(table(df$ancestry, df$case))
  
  message("Age Summary:")
  print(summary(df$age))
  message("---------------------")
  
  # 5) Return the combined data frame
  return(df)
}

In [ ]:
# Call the function 
polio_combined <- prepare_case_control_data(
  cases_df = polio, 
  controls_df = polio_ctrl
)


# You can now work with your new data frame
# head(polio_combined)

In [ ]:
hiv_combined <- prepare_case_control_data(
  cases_df = hiv, 
  controls_df = hiv_ctrl
)

In [ ]:
#### run matchit on merged wrangled dataframes

In [ ]:

perform_matching <- function(input_data, 
                             match_ratio = 6, 
                             exact_vars = c("sex", "ancestry")) {
  
  # 1) Run the matching
  # The formula case ~ age + sex + ancestry is hard-coded as requested
  m.out <- MatchIt::matchit(
    case ~ age + sex + ancestry,
    data = input_data,
    method = "nearest",           # Use nearest neighbor...
    exact = exact_vars,           # ...but ONLY within exact groups
    ratio = match_ratio,          # Desired controls per case
    distance = "mahalanobis"
  )
  
  # 2) Print diagnostics to the console
  message("--- Matching Summary ---")
  print(summary(m.out))
  
  message("\n--- Balance Plot (Love Plot) ---")
  # Use tryCatch in case 'cobalt' isn't loaded or plot fails
  tryCatch({
    # We must explicitly 'print()' a plot when inside a function
    print(cobalt::love.plot(m.out))
  }, error = function(e) {
    message("Could not generate love.plot. Error: ", e$message)
  })
  
  # 3) Get matched/weighted cohorts
  matched_data <- MatchIt::match.data(m.out)
  
  # 4) Return the final matched data frame
  return(matched_data)
}

In [ ]:
#  Call the function and store the result
# The function will print the summary and plot
polio_matched_cohort <- perform_matching(input_data = polio_combined)

# 'matched_cohort' now holds your final matched data
head(polio_matched_cohort)

# ---
# You can also easily change the ratio:
# matched_5_to_1 <- perform_matching(input_data = df, match_ratio = 5)

In [ ]:
#  Call the function and store the result
# The function will print the summary and plot
hiv_matched_cohort <- perform_matching(input_data = hiv_combined)

# 'matched_cohort' now holds your final matched data
head(hiv_matched_cohort)

# ---
# You can also easily change the ratio:
# matched_5_to_1 <- perform_matching(input_data = df, match_ratio = 5)

In [ ]:
head(matched_cohort)

In [ ]:
tail(matched_cohort)

In [ ]:
colnames(matched_cohort)

In [ ]:
write.csv(matched, "cohort_data/matched_polio_case_control_cohorts.csv", row.names = FALSE)
